In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pib_helper import load_psi4_molecule, calculate_r_squared
import psi4

psi4.set_output_file('output.dat', False)
psi4.set_memory('1 GB')

<a id="part4"></a>

## Part 4 — Comparing Models

We now have two models for the electronic structure of conjugated polyenes:
- The **particle-in-a-box (PIB)** — a simple analytic formula with no adjustable parameters.
- **Hartree–Fock (HF)** — a detailed quantum-chemical calculation.

How do they compare to each other, and to **experiment**? And can we improve a simple model by fitting it to match experimental data?

This is a common strategy in computational chemistry: use a cheap model with empirically fitted parameters to approximate expensive calculations or experiments. If the cheap model captures the right *shape* of the relationship, a linear correction can bring the absolute values into agreement:

$$E_{\text{corrected}} = \alpha \cdot E_{\text{model}} + \beta$$

### Polyenes for this study

| $n_c$ | Name | XYZ file |
|-------|------|----------|
| 4 | butadiene | `data/butadiene.xyz` |
| 6 | hexatriene | `data/hexatriene.xyz` |
| 8 | octatetraene | `data/octatetraene.xyz` |
| 10 | decapentaene | `data/decapentaene.xyz` |
| 12 | dodecahexaene | `data/dodecahexaene.xyz` |
| 14 | tetradecaheptaene | `data/tetradecaheptaene.xyz` |
| 16 | hexadecaoctaene | `data/hexadecaoctaene.xyz` |

### Key formulas

- Total electrons: $n_e = 7 n_c + 2$
- HOMO index (zero-indexed): $n_e / 2 - 1$
- Conversion: 1 Hartree = 27.211 eV

---

### Coding Activity 4
`20 points`

- C4.3: Apply the particle-in-a-box model to conjugated polyenes.
- C4.4: Evaluate the tradeoffs of different physical models.
- P4.2: Fit nonlinear curves to data.

#### Part A — HF HOMO–LUMO Gaps

The cell below runs a Hartree–Fock calculation on **ethane** and extracts the HOMO–LUMO gap. Study it carefully — you will adapt this approach for the polyenes.

Ethane (C$_2$H$_6$) has $n_e = 18$ electrons, so the HOMO is at orbital index 8 (zero-indexed: $18/2 - 1 = 8$). The molecular geometry is loaded from a pre-built XYZ file using `load_psi4_molecule()`.

In [ ]:
# --- Worked example: ethane HOMO-LUMO gap ---
mol_ethane = load_psi4_molecule('data/ethane.xyz')
energy_ethane, wfn_ethane = psi4.energy('SCF/STO-3G', return_wfn=True, molecule=mol_ethane)

eps_ethane = wfn_ethane.epsilon_a().np
n_e_ethane = 18
homo_idx = n_e_ethane // 2 - 1
lumo_idx = homo_idx + 1
gap_ethane = (eps_ethane[lumo_idx] - eps_ethane[homo_idx]) * 27.211  # convert to eV

print(f'Ethane HF/STO-3G energy: {energy_ethane:.6f} a.u.')
print(f'HOMO index: {homo_idx}, LUMO index: {lumo_idx}')
print(f'HOMO-LUMO gap: {gap_ethane:.2f} eV')

In [ ]:
# Subgoal: Compute HF HOMO-LUMO gaps for the polyenes

polyene_nc = [4, 6, 8, 10, 12, 14, 16]
polyene_xyz = [
    'data/butadiene.xyz',
    'data/hexatriene.xyz',
    'data/octatetraene.xyz',
    'data/decapentaene.xyz',
    'data/dodecahexaene.xyz',
    'data/tetradecaheptaene.xyz',
    'data/hexadecaoctaene.xyz',
]

hf_gaps = []

# Subgoal: Loop over polyenes and compute HF gaps
for n_c, xyz_file in zip(polyene_nc, polyene_xyz):
    mol = load_psi4_molecule(xyz_file)
    energy, wfn = psi4.energy('SCF/STO-3G', return_wfn=True, molecule=mol)
    eps = wfn.epsilon_a().np
    n_e = 7 * n_c + 2
    homo_idx = n_e // 2 - 1
    lumo_idx = homo_idx + 1
    gap = (eps[lumo_idx] - eps[homo_idx]) * 27.211
    hf_gaps.append(gap)
    print(f'n_c={n_c}: E={energy:.6f} a.u., gap={gap:.2f} eV')

print('HF HOMO-LUMO gaps (eV):', hf_gaps)

In [ ]:
# Subgoal: Plot HF gaps vs chain length

plt.figure(figsize=(8, 5))
plt.plot(polyene_nc, hf_gaps, 's-', color='steelblue', label='HF/STO-3G')
plt.xlabel('Number of carbon atoms')
plt.ylabel('HOMO-LUMO gap (eV)')
plt.title('HF HOMO-LUMO Gaps for Conjugated Polyenes')
plt.legend()
plt.show()

#### Part B — PIB HOMO–LUMO Gaps

The particle-in-a-box model gives the energy of level $n$ as:

$$E_n = \frac{n^2 \pi^2}{2 L^2} \quad \text{(atomic units)}$$

For a polyene with $n_c$ carbon atoms:
- Box length: $L = 1.4 \times (n_c - 1)$ Å, converted to a.u. by multiplying by 1.8897
- HOMO quantum number: $n_{\text{HOMO}} = n_c / 2$
- LUMO quantum number: $n_{\text{LUMO}} = n_c / 2 + 1$
- Gap: $\Delta E = E_{\text{LUMO}} - E_{\text{HOMO}}$, converted to eV by multiplying by 27.211

In [ ]:
# Subgoal: Write a function that returns the PIB HOMO-LUMO gap in eV

def pib_energy_gap(n_c):
    """
    Compute the PIB HOMO-LUMO gap for a polyene with n_c carbons.
    
    Parameters
    ----------
    n_c : int
        Number of carbon atoms.
    
    Returns
    -------
    gap_eV : float
        HOMO-LUMO gap in eV.
    """
    # Subgoal: Compute box length in atomic units
    L = 1.4 * (n_c - 1)       # Angstroms
    L_au = L * 1.8897          # atomic units
    
    # Subgoal: Compute HOMO and LUMO energies
    n_homo = n_c // 2
    n_lumo = n_c // 2 + 1
    E_homo = n_homo**2 * np.pi**2 / (2 * L_au**2)
    E_lumo = n_lumo**2 * np.pi**2 / (2 * L_au**2)
    
    # Subgoal: Return gap in eV
    gap_eV = (E_lumo - E_homo) * 27.211
    return gap_eV

In [ ]:
# Subgoal: Compute PIB gaps and plot alongside HF gaps

pib_gaps = [pib_energy_gap(n_c) for n_c in polyene_nc]
print('PIB HOMO-LUMO gaps (eV):', [f'{g:.2f}' for g in pib_gaps])

# Subgoal: Plot both models
plt.figure(figsize=(8, 5))
plt.plot(polyene_nc, hf_gaps, 's-', color='steelblue', label='HF/STO-3G')
plt.plot(polyene_nc, pib_gaps, 'o-', color='darkorange', label='PIB')
plt.xlabel('Number of carbon atoms')
plt.ylabel('HOMO-LUMO gap (eV)')
plt.title('HF vs PIB HOMO-LUMO Gaps')
plt.legend()
plt.show()

#### Part C — Comparison to Experiment

We have experimental UV absorption data for these polyenes. Let's see how both models compare to reality.

We'll also apply a **linear correction** to each model:

$$E_{\text{corrected}} = \alpha \cdot E_{\text{model}} + \beta$$

This fits two parameters ($\alpha$, $\beta$) to best match the experimental data. If the model captures the right *trend*, the correction should give good agreement.

In [ ]:
# --- Given: load experimental data ---
expt_data = pd.read_csv('data/polyene_excitation.csv')
print(expt_data)

expt_energies = expt_data['excitation_energy_eV'].values

In [ ]:
# Subgoal: Plot all three datasets on the same axes

plt.figure(figsize=(8, 5))
plt.plot(polyene_nc, hf_gaps, 's-', color='steelblue', label='HF/STO-3G')
plt.plot(polyene_nc, pib_gaps, 'o-', color='darkorange', label='PIB')
plt.plot(polyene_nc, expt_energies, 'D-', color='green', label='Experiment')
plt.xlabel('Number of carbon atoms')
plt.ylabel('Excitation energy (eV)')
plt.title('Model vs Experimental Excitation Energies')
plt.legend()
plt.show()

In [ ]:
# Subgoal: Fit linear corrections to both models

def linear_model(x, alpha, beta):
    """Linear correction: alpha * x + beta."""
    return alpha * x + beta

# Subgoal: Fit HF model to experiment
popt_hf, _ = curve_fit(linear_model, hf_gaps, expt_energies)
print(f'HF correction:  alpha = {popt_hf[0]:.4f}, beta = {popt_hf[1]:.4f}')

# Subgoal: Fit PIB model to experiment
popt_pib, _ = curve_fit(linear_model, pib_gaps, expt_energies)
print(f'PIB correction: alpha = {popt_pib[0]:.4f}, beta = {popt_pib[1]:.4f}')

In [ ]:
# Subgoal: Compare corrected predictions to experiment

# Subgoal: Compute corrected predictions
hf_corrected = linear_model(np.array(hf_gaps), *popt_hf)
pib_corrected = linear_model(np.array(pib_gaps), *popt_pib)

# Subgoal: Plot corrected models vs experiment
plt.figure(figsize=(8, 5))
plt.plot(polyene_nc, expt_energies, 'D-', color='green', label='Experiment')
plt.plot(polyene_nc, hf_corrected, 's--', color='steelblue', label='HF corrected')
plt.plot(polyene_nc, pib_corrected, 'o--', color='darkorange', label='PIB corrected')
plt.xlabel('Number of carbon atoms')
plt.ylabel('Excitation energy (eV)')
plt.title('Corrected Models vs Experiment')
plt.legend()
plt.show()

# Subgoal: Compute R-squared for each
r2_hf = calculate_r_squared(expt_energies, hf_corrected)
r2_pib = calculate_r_squared(expt_energies, pib_corrected)
print(f'R² (HF corrected):  {r2_hf:.4f}')
print(f'R² (PIB corrected): {r2_pib:.4f}')

### Question 4
`10 points`

- C4.3: Apply the particle-in-a-box model to conjugated polyenes.
- C4.4: Evaluate the tradeoffs of different physical models.

a) Before correction: which model (HF or PIB) is closer to the experimental values? Is either one quantitatively accurate?

b) After the linear correction: compare the $R^2$ values. What does this tell you about how well each model captures the *trend* in excitation energies?

c) What are the advantages of using a cheap fitted model (like corrected PIB) compared to running a full HF calculation every time?

---

*Your answer here (`double click me!`):*

**Model answers:**

a) Neither model is quantitatively accurate before correction. The HF HOMO–LUMO gaps overestimate excitation energies (Koopmans' theorem gaps are too large), while the PIB gaps have the right order of magnitude but use a very crude approximation for the molecular potential. HF is generally closer in absolute values because it accounts for electron–electron repulsion and realistic geometry.

b) Both models should have high $R^2$ values after linear correction, indicating that both capture the correct *trend* — excitation energy decreases with increasing conjugation length. With only 4 data points, even a rough model can achieve a good linear fit, so the $R^2$ values mainly confirm that both models get the qualitative shape right.

c) The PIB model requires only a pocket calculator — no quantum chemistry software, no 3D geometry, no iterative convergence. Once the linear correction parameters are fitted, predictions for new chain lengths are instantaneous. This makes it useful for quick estimates, screening large numbers of molecules, or building intuition about trends, even though it sacrifices quantitative accuracy for individual molecules.

---

<a id="reflection"></a>

## Reflection
`10 points`

a) In this lab, you worked with models at several levels of detail: a simple analytic formula (PIB), a numerical variational procedure, and a full quantum-chemical method (Hartree–Fock). In your own words, what makes a model "useful" even when it is known to be imperfect?

b) Describe one situation in this lab where a numerical procedure (optimization, curve fitting, or iteration) was essential — where you could not have obtained the result by hand calculation alone. What did you learn from this?

c) What is one thing from this lab that you found surprising or that changed how you think about quantum mechanics or computational chemistry?

*Your answer here (`double click me!`):*